# Parse UE docs TOC HTML into URLs

Reads `scraped_data/urls/full_ur.txt` (saved Epic contents-table HTML), walks the nested `contents-table-list` tree to build breadcrumb paths (e.g. `Understanding the Basics > Levels > World Settings`), and writes:

- `scraped_data/urls/full_ur_with_paths.json` — JSON array: `title`, `url`, `path`, `path_segments`, `scrape_url`
- `scraped_data/urls/full_ur_parsed.jsonl` — one `{"title","url"}` per line (unchanged shape for older scripts)
- `scraped_data/urls/full_ur_urls_5_7.txt` — one canonical URL per line with `?application_version=…` (filename tracks `UE_VERSION`)

In [ ]:
import json
import re
from pathlib import Path
from urllib.parse import parse_qs, urlencode, urlparse, urlunparse, urljoin

from bs4 import BeautifulSoup

HERE = Path(".").resolve()
HTML_PATH = HERE / "scraped_data" / "urls" / "full_ur.txt"
OUT_JSON = HERE / "scraped_data" / "urls" / "full_ur_with_paths.json"
OUT_JSONL = HERE / "scraped_data" / "urls" / "full_ur_parsed.jsonl"
UE_VERSION = "5.7"
OUT_TXT = HERE / "scraped_data" / "urls" / f"full_ur_urls_{UE_VERSION.replace('.', '_')}.txt"
BASE = "https://dev.epicgames.com"


def clean_text(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def normalize_doc_url(url: str) -> str | None:
    parsed = urlparse(url)
    if "dev.epicgames.com" not in parsed.netloc and parsed.netloc != "":
        return None
    path = parsed.path
    if not path.startswith("/documentation/en-us/unreal-engine"):
        return None
    qs = parse_qs(parsed.query)
    new_qs = {}
    if "application_version" in qs:
        new_qs["application_version"] = qs["application_version"][0]
    query_str = urlencode(new_qs) if new_qs else ""
    return urlunparse(("https", "dev.epicgames.com", path, "", query_str, ""))


def url_with_application_version(url: str, version: str) -> str:
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    qs["application_version"] = [version]
    flat = {k: v[0] for k, v in qs.items()}
    return urlunparse(("https", parsed.netloc, parsed.path, "", urlencode(flat), ""))


def _segment_from_div(div) -> str | None:
    if not div:
        return None
    a = div.find("a", class_="contents-table-link", recursive=False)
    if a:
        return clean_text(a.get_text()) or None
    sp = div.find("span", recursive=False)
    if sp:
        t = sp.get("title")
        if t:
            return clean_text(t) or None
        return clean_text(sp.get_text()) or None
    return None


def _walk_contents_ul(ul, path_stack: list[str], rows: list[dict], seen: set[str], *, base: str) -> None:
    for li in ul.find_all("li", class_="contents-table-item", recursive=False):
        div = li.find("div", class_="contents-table-el", recursive=False)
        seg = _segment_from_div(div)
        child_stack = path_stack + ([seg] if seg else [])
        if div:
            a = div.find("a", class_="contents-table-link", recursive=False)
            if a and a.get("href"):
                href = (a.get("href") or "").strip()
                if href:
                    abs_url = urljoin(base, href)
                    title = clean_text(a.get_text())
                    if not title:
                        title = abs_url.rstrip("/").split("/")[-1] or abs_url
                    parts = path_stack + [title]
                    path_str = " > ".join(parts)
                    if abs_url not in seen:
                        seen.add(abs_url)
                        rows.append(
                            {
                                "title": title,
                                "url": abs_url,
                                "path": path_str,
                                "path_segments": parts,
                            }
                        )
        nested = li.find("ul", class_="contents-table-list", recursive=False)
        if nested:
            _walk_contents_ul(nested, child_stack, rows, seen, base=base)


def parse_epic_contents_table_with_paths(html: str, *, base: str = BASE) -> list[dict]:
    """Walk nested TOC; each unique ``a.contents-table-link[href]`` gets a breadcrumb path."""
    soup = BeautifulSoup(html, "html.parser")
    rows: list[dict] = []
    seen: set[str] = set()
    root_stack: list[str] = []
    root_div = soup.select_one("div.contents-table-el.is-root-entry")
    if root_div:
        ra = root_div.find("a", class_="contents-table-link", recursive=False)
        if ra and ra.get("href"):
            href = (ra.get("href") or "").strip()
            if href:
                abs_url = urljoin(base, href)
                title = clean_text(ra.get_text()) or abs_url.rstrip("/").split("/")[-1]
                parts = [title]
                if abs_url not in seen:
                    seen.add(abs_url)
                    rows.append(
                        {"title": title, "url": abs_url, "path": title, "path_segments": parts}
                    )
                root_stack = [title]
    root_ul = soup.find("ul", class_="contents-table-list")
    if root_ul:
        _walk_contents_ul(root_ul, root_stack, rows, seen, base=base)
    return rows


def urls_for_scraping(entries: list[dict], version: str) -> list[str]:
    urls: list[str] = []
    seen: set[str] = set()
    for row in entries:
        n = normalize_doc_url(row["url"])
        if n is None:
            continue
        u = url_with_application_version(n, version)
        if u in seen:
            continue
        seen.add(u)
        urls.append(u)
    return urls


html = HTML_PATH.read_text(encoding="utf-8")
rows = parse_epic_contents_table_with_paths(html)
OUT_JSON.parent.mkdir(parents=True, exist_ok=True)

full_rows: list[dict] = []
for row in rows:
    n = normalize_doc_url(row["url"])
    scrape_u = url_with_application_version(n, UE_VERSION) if n else ""
    full_rows.append({**row, "scrape_url": scrape_u})

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(full_rows, f, ensure_ascii=False, indent=2)

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps({"title": row["title"], "url": row["url"]}, ensure_ascii=False) + "\n")

scrape_urls = urls_for_scraping(rows, UE_VERSION)
OUT_TXT.write_text("\n".join(scrape_urls) + "\n", encoding="utf-8")

print(f"HTML: {HTML_PATH} ({len(html):,} chars)")
print(f"Parsed entries: {len(rows)}")
print(f"Canonical URLs for UE {UE_VERSION}: {len(scrape_urls)}")
print(f"Wrote {OUT_JSON}")
print(f"Wrote {OUT_JSONL}")
print(f"Wrote {OUT_TXT}")